<a href="https://colab.research.google.com/github/ronyates47/Gedcom-Utils/blob/main/anchor_engine_v3_base_runall.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
# @title [CELL 1] Environment Setup & Configuration
print("="*60)
print("      [CELL 1] SYSTEM WAKE-UP & ENVIRONMENT SETUP...")
print("="*60)

# 1. Install required packages (ensures Colab has what we need)
!pip install -q pandas pytz

# 2. Import standard libraries used across the entire pipeline
import os
import re
import csv
import json
import math
import shutil
import pandas as pd
import pytz
from datetime import datetime
from ftplib import FTP_TLS

# 3. Securely load FTP Credentials from Colab Secrets
try:
    from google.colab import userdata
    print("\n[+] Google Colab Environment Detected. Loading secrets...")

    # Check and set FTP Host
    HOST = os.environ.get("FTP_HOST") or userdata.get("FTP_HOST")
    if HOST:
        os.environ["FTP_HOST"] = HOST
        print("    ✅ FTP_HOST loaded.")
    else:
        print("    ⚠️ WARNING: FTP_HOST not found in secrets.")

    # Check and set FTP User
    USER = os.environ.get("FTP_USER") or userdata.get("FTP_USER")
    if USER:
        os.environ["FTP_USER"] = USER
        print("    ✅ FTP_USER loaded.")
    else:
        print("    ⚠️ WARNING: FTP_USER not found in secrets.")

    # Check and set FTP Password
    PASS = os.environ.get("FTP_PASS") or userdata.get("FTP_PASS")
    if PASS:
        os.environ["FTP_PASS"] = PASS
        print("    ✅ FTP_PASS loaded.")
    else:
        print("    ⚠️ WARNING: FTP_PASS not found in secrets.")

except ImportError:
    print("\n[!] Running outside of Google Colab. Make sure environment variables are set manually.")
except userdata.SecretNotFoundError as e:
    print(f"\n❌ SECRET ERROR: {e}")
    print("    Please click the '🔑 Keys' icon on the left sidebar and ensure FTP_HOST, FTP_USER, and FTP_PASS are saved.")

print("\n✅ Cell 1 (Environment Setup) Complete. The system is ready.")
print("="*60)

      [CELL 1] SYSTEM WAKE-UP & ENVIRONMENT SETUP...

[+] Google Colab Environment Detected. Loading secrets...
    ✅ FTP_HOST loaded.
    ✅ FTP_USER loaded.
    ✅ FTP_PASS loaded.

✅ Cell 1 (Environment Setup) Complete. The system is ready.


In [16]:
# @title [CELL 2] The Data Interceptor (Smart GEDCOM Fetch)
import os
import glob
from ftplib import FTP_TLS

print("="*60)
print("      📥 PRE-LOADING DATA (SMART FETCH)...")
print("="*60)

# 1. LOCAL CACHE CHECK: Do we already have a GEDCOM?
local_geds = glob.glob("*.ged")
gedcom_needed = True

if local_geds:
    print(f"[+] Local cache found: {local_geds[0]}. Skipping FTP download for GEDCOM.")
    gedcom_needed = False
    gedcom_name = local_geds[0]

try:
    # Uses the secrets already loaded by Cell 1
    ftps = FTP_TLS()
    ftps.connect(HOST, 21); ftps.auth(); ftps.login(USER, PASS); ftps.prot_p()

    # 2. FETCH NEWEST GEDCOM FROM TNG
    if gedcom_needed:
        print("[+] Searching for the newest GEDCOM in /tng/gedcom/...")

        # Navigate to the TNG folder from the root login directory
        try:
            ftps.cwd("/tng/gedcom")
        except:
            ftps.cwd("tng/gedcom") # Fallback for relative path

        files = ftps.nlst()
        ged_files = [f for f in files if f.lower().endswith('.ged')]

        if not ged_files:
            print("    ❌ ERROR: No .ged files found in /tng/gedcom/")
        else:
            newest_ged = None
            newest_time = ""

            # Ask the server for the exact modification time of every file
            for ged in ged_files:
                try:
                    res = ftps.voidcmd(f"MDTM {ged}")
                    mtime = res[4:].strip() # Returns format like 20260302123000
                    if mtime > newest_time:
                        newest_time = mtime
                        newest_ged = ged
                except:
                    pass # Skip if server denies MDTM for a specific file

            # Fallback to alphabetical sorting if server time-check fails
            if not newest_ged:
                newest_ged = sorted(ged_files)[-1]

            print(f"    [+] Newest GEDCOM identified: {newest_ged}")
            print(f"    [+] Downloading {newest_ged} to Colab memory...")

            with open(newest_ged, "wb") as f:
                ftps.retrbinary(f"RETR {newest_ged}", f.write)
            print("    ✅ GEDCOM successfully loaded.")

    # 3. FETCH CSV FROM THE ANCHOR SPOKE
    print("\n[+] Downloading engine_database.csv from Anchor Spoke...")

    # Jump back to the root directory, then into the Anchor folder
    ftps.cwd("/")
    try:
        ftps.cwd("anchor/anc-1000-yates")
    except:
        ftps.cwd("/anchor/anc-1000-yates")

    with open("engine_database.csv", "wb") as f:
        ftps.retrbinary("RETR engine_database.csv", f.write)
    print("    ✅ CSV successfully loaded.")

    ftps.quit()
    print("\n🎉 Smart Fetch Complete! Cell 2 is ready to run.")

except Exception as e:
    print(f"\n❌ FTP Download Failed: {e}")

      📥 PRE-LOADING DATA (SMART FETCH)...
[+] Local cache found: yates_study_2025.ged. Skipping FTP download for GEDCOM.

[+] Downloading engine_database.csv from Anchor Spoke...
    ✅ CSV successfully loaded.

🎉 Smart Fetch Complete! Cell 2 is ready to run.


In [17]:
# @title [CELL 3] The Data & Math Engine (Zero-Match Injection Update)
print("="*60)
print("      [CELL 2] DATA & MATH ENGINE STARTING...")
print("="*60)

import os, sys, re, csv, json, math, shutil
import pandas as pd
from ftplib import FTP_TLS
try:
    from google.colab import userdata
except ImportError:
    pass

CSV_DB = "engine_database.csv"
JSON_DB = "compiled_database.json"
KEY_FILE = "match_to_unmasked.csv"
PROCESSED_GED = "_processed_unmasked.ged"

try:
    HOST = os.environ.get("FTP_HOST") or userdata.get("FTP_HOST")
    USER = os.environ.get("FTP_USER") or userdata.get("FTP_USER")
    PASS = os.environ.get("FTP_PASS") or userdata.get("FTP_PASS")
except: pass

# --- STEP 1: FETCH FILES ---
if not os.path.exists(KEY_FILE):
    print(f"    [+] Fetching {KEY_FILE} via FTP...")
    try:
        ftps = FTP_TLS(); ftps.connect(HOST, 21); ftps.auth(); ftps.login(USER, PASS); ftps.prot_p()
        with open(KEY_FILE, "wb") as f: ftps.retrbinary(f"RETR /anchor/anc-1000-yates/{KEY_FILE}", f.write)
        ftps.quit()
    except Exception as e: print(f"    ⚠️ FTP fetch failed: {e}")

ged_files = [f for f in os.listdir('.') if f.lower().endswith('.ged') and "_processed" not in f.lower()]
if not ged_files:
    print("❌ No GEDCOM found. Please upload one.")
else:
    DEFAULT_GEDCOM = sorted(ged_files, key=lambda x: os.path.getmtime(x), reverse=True)[0]
    shutil.copyfile(DEFAULT_GEDCOM, PROCESSED_GED)
    print(f"    [+] Active GEDCOM: {DEFAULT_GEDCOM}")

# --- STEP 2: LOAD AUTH & PARSE GEDCOM ---
csv_auth = {}
if os.path.exists(KEY_FILE):
    with open(KEY_FILE, 'r', errors='replace') as f:
        for i, row in enumerate(csv.reader(f)):
            if i > 0 and len(row) >= 2:
                code, name = row[0].strip().lower(), row[1].strip()
                tid = "I" + re.sub(r'[^0-9]', '', row[2]) if len(row)>2 and row[2].strip() else ""
                sort_key = row[3].strip().lower() if len(row)>3 else ""
                csv_auth[code] = {"name": name, "id": tid, "sort_key": sort_key}

individuals = {}; families = {}; persons_data = {}
current_id = None; current_fam = None; mode = None

def clean_name(n):
    if not n: return "findme"
    s = n.replace("/", "").strip()
    if s.lower() in ["unknown", "missing", "searching", "living", "private", "nee", "wife"] or "?" in s: return "findme"
    return s

with open(PROCESSED_GED, "r", encoding="utf-8", errors="replace") as f:
    for line in f:
        line = line.strip(); parts = line.split(" ", 2)
        if len(parts) < 2: continue
        lvl, tag, val = parts[0], parts[1], parts[2] if len(parts)>2 else ""

        if lvl == "0" and "INDI" in val:
            current_id = tag.replace("@", "")
            individuals[current_id] = {"name": "findme", "famc": None, "fams": [], "code": "", "cm": 0}
            persons_data[current_id] = {'name': f'ID {current_id}', 'bdate': '', 'bplace': '', 'ddate': '', 'dplace': '', 'sources_count': 0, 'citations_count': 0}
            current_fam = None; mode = None
        elif current_id and lvl != "0":
            if tag == "NAME" and lvl == "1":
                individuals[current_id]["name"] = clean_name(val)
                persons_data[current_id]["name"] = val.replace('/', '').strip()
            elif tag == "FAMC" and lvl == "1": individuals[current_id]["famc"] = val.replace("@", "")
            elif tag == "FAMS" and lvl == "1": individuals[current_id]["fams"].append(val.replace("@", ""))
            elif tag == "NPFX" and lvl == "2":
                m_code = re.search(r'(\d+)\s*&?\s*([^ \t\n\r\f\v]+)', val)
                if m_code: individuals[current_id]["code"] = m_code.group(2).lower()
                m_cm = re.search(r'^(\d+)|(\d+)\s*cM', val, re.IGNORECASE)
                if m_cm: individuals[current_id]["cm"] = int(m_cm.group(1) or m_cm.group(2))
            elif tag == "BIRT": mode = "BIRT"
            elif tag == "DEAT": mode = "DEAT"
            elif tag == "SOUR" and lvl == "1": persons_data[current_id]['sources_count'] += 1; mode = None
            elif tag == "SOUR" and lvl == "2": persons_data[current_id]['citations_count'] += 1
            elif tag == "DATE" and mode == "BIRT": persons_data[current_id]['bdate'] = val
            elif tag == "DATE" and mode == "DEAT": persons_data[current_id]['ddate'] = val
            elif tag == "PLAC" and mode == "BIRT": persons_data[current_id]['bplace'] = val
            elif tag == "PLAC" and mode == "DEAT": persons_data[current_id]['dplace'] = val
            elif lvl == "1": mode = None

        elif lvl == "0" and "FAM" in val:
            current_fam = tag.replace("@", "")
            families[current_fam] = {"husb": None, "wife": None}
            current_id = None
        elif current_fam and lvl != "0":
            if tag == "HUSB": families[current_fam]["husb"] = val.replace("@", "")
            elif tag == "WIFE": families[current_fam]["wife"] = val.replace("@", "")

# --- STEP 3: BUILD BASE CSV ROWS ---
print("    [+] Tracing Lineages & Generating CSV...")
def get_parents(pid):
    if not pid or pid not in individuals: return None, None
    famc = individuals[pid]["famc"]
    if not famc or famc not in families: return None, None
    return families[famc]["husb"], families[famc]["wife"]

yates_memo = {}
def has_yates(pid):
    if not pid or pid not in individuals: return False
    if pid in yates_memo: return yates_memo[pid]
    n = individuals[pid]["name"].lower()
    if "yates" in n or "yeates" in n: yates_memo[pid] = True; return True
    d, m = get_parents(pid)
    res = has_yates(d) or has_yates(m)
    yates_memo[pid] = res; return res

def climb(start_id):
    curr = start_id; lin = []
    while curr:
        p = individuals.get(curr)
        if not p: break
        spouse_name = "findme"; sp_id = None
        if p["fams"]:
            fid = p["fams"][0]
            if fid in families:
                f = families[fid]
                sp_id = f["wife"] if f["husb"] == curr else f["husb"]
                if sp_id and sp_id in individuals: spouse_name = individuals[sp_id]["name"]
        lin.append({"name": p["name"], "id": curr, "spouse": spouse_name, "sp_id": sp_id})
        d, m = get_parents(curr)
        if not d and not m: break
        dy, my = has_yates(d), has_yates(m)
        curr = d if dy and not my else (m if my and not dy else (d if d else m))
    return lin

for code, td in csv_auth.items():
    if td["id"] and td["id"] in individuals:
        full = list(reversed(climb(td["id"])))
        td["lin_str"] = " -> ".join([x["name"] for x in full])
        td["pids"] = ",".join([x["id"] for x in full])
    else:
        td["lin_str"] = ""; td["pids"] = ""

rows = []
for uid, p in individuals.items():
    if p["code"]:
        kc = p["code"]
        td = csv_auth.get(kc, {"name": kc, "id": "", "sort_key": "", "lin_str": "", "pids": ""})
        t_disp = f"{td['name']} [{td['id']}]" if td['id'] else f"{td['name']} [{kc}]"

        lin = climb(uid)
        if not lin: continue
        full = list(reversed(lin))

        gen1 = full[0]
        top_name = gen1["name"]; sp_name = gen1["spouse"]
        pair_simp = f"{top_name} & {sp_name}" if sp_name != "findme" else top_name

        sur = top_name.split()[-1] if top_name.split() else ""
        firsts = re.sub(f"{re.escape(sur)}$", "", top_name).strip() if sur else top_name
        b_yr = re.search(r'\d{4}', persons_data[gen1['id']]['bdate'])
        d_yr = re.search(r'\d{4}', persons_data[gen1['id']]['ddate'])
        dates = f"({b_yr.group(0) if b_yr else 'findme'} - {d_yr.group(0) if d_yr else 'findme'})"
        dir_lbl = f"{sur}, {firsts} {dates}" + (f" & {sp_name}" if sp_name != "findme" else "")

        rows.append({
            "Tester_Code": kc, "Tester_Name": td["name"], "Tester_ID": td["id"], "Tester_Display": t_disp,
            "Tester_Sort_Key": td["sort_key"], "Tester_Lineage": td["lin_str"], "Tester_Path_IDs": td["pids"],
            "Match_Name": p["name"], "Match_ID": uid, "cM": p["cm"],
            "Match_Lineage": " -> ".join([pair_simp] + [x["name"] for x in full[1:]]),
            "Match_Path_IDs": ",".join([x["id"] for x in full]),
            "Authority_Directory_Label": dir_lbl
        })

df = pd.DataFrame(rows)
df.to_csv(CSV_DB, index=False, encoding="iso-8859-15", quoting=csv.QUOTE_ALL)
df.rename(columns={"Authority_Directory_Label": "Dir_Label", "Tester_Code": "Kit_Code", "Match_Lineage": "Lineage", "Match_Path_IDs": "s_ids", "Tester_Display": "Kit_Name"}, inplace=True)
df['search_ids'] = df['s_ids']; df['search_names'] = df['Lineage'].astype(str).str.replace(' -> ', '|')
df['t_names'] = df['Tester_Lineage'].astype(str).str.replace(' -> ', '|'); df['t_ids'] = df['Tester_Path_IDs'].astype(str).str.replace(',', '|')

# --- STEP 4: MATHEMATICAL ENGINE (ANCHOR/CSS) ---
print("    [+] Calculating Math Matrices & Audits...")
def clean_num(s): return re.sub(r'[^0-9]', '', str(s))
def norm_log(val, cap): return min(1.0, math.log(1 + max(0, val or 0)) / math.log(1 + cap))

id_to_kits = {}
for _, r in df.iterrows():
    for i in [clean_num(x) for x in str(r['s_ids']).split(',') if x]:
        id_to_kits.setdefault(i, set()).add(r['Kit_Name'])

participant_scores = []
for p_name, grp in df.groupby('Kit_Name'):
    val_grp = grp[grp['Dir_Label'] != 'No Matches']
    pm = len(val_grp)
    if pm == 0: continue

    dc = val_grp['Dir_Label'].value_counts()
    hc_t, hc_2 = (int(dc.iloc[0]), int(dc.iloc[1])) if len(dc)>1 else (int(dc.iloc[0]), 0) if len(dc)>0 else (0,0)
    target_anc = str(dc.index[0]) if len(dc)>0 else "Unknown"

    hh, target_id = 0, None
    for _, r in val_grp.iterrows():
        for i in [clean_num(x) for x in str(r['s_ids']).split(',') if x]:
            if len(id_to_kits.get(i, set())) > hh: hh, target_id = len(id_to_kits[i]), i

    spine_ids = []; br = 0; tb = 0
    if target_id:
        tb = len(id_to_kits.get(target_id, set()))
        col = df[df['s_ids'].apply(lambda x: target_id in [clean_num(y) for y in str(x).split(',') if y])]
        b_set = set()
        for _, r in col.iterrows():
            ids = [clean_num(x) for x in str(r['s_ids']).split(',') if x]
            nms = str(r['search_names']).split('|')
            if target_id in ids:
                idx = ids.index(target_id)
                b_set.add(nms[idx+1].replace('findme', '?').replace('FINDME', '?').split(' (')[0].strip() if idx+1 < len(nms) else "Direct")
        br = len(b_set)
        spine_ids = [clean_num(x) for x in str(val_grp[val_grp['s_ids'].apply(lambda x: target_id in str(x))].iloc[0]['s_ids']).split(',') if x]

    # Math formulas
    dr = float(hc_t / max(1, hc_2))
    w_sum = norm_log(pm, 150) + norm_log(hc_t, 100) + (norm_log(dr, 10)*1.5) + norm_log(tb, 40) + norm_log(hh, 150)
    w_sum += 2.0 * (1.0 if br>=6 else 0.85 if br==5 else 0.70 if br==4 else 0.50 if br==3 else 0.25 if br==2 else 0)
    st_str, st_val = ("PASS", 1.0) if pm>=15 and br>=3 and dr>=1.5 else ("PARTIAL", 0.85) if pm>=15 and br>=2 else ("FAIL", 0.6)
    css = float(100 * (w_sum / 7.5) * st_val)

    # Doc Audit
    tp_viol = 0; tp_checks = 0; diag_log = []
    for pid in spine_ids:
        d = persons_data.get(pid, {})
        if d.get('sources_count',0) + d.get('citations_count',0) == 0: diag_log.append(f"Missing Sources: {d.get('name')} (I{pid})")

    docs_score = 88.5 if len(diag_log)==0 else max(35.0, 85.0 - (len(diag_log)*5))
    anchor = min(100.0, (0.65 * css) + (0.35 * docs_score) + (10.0 * min(css, docs_score)/100.0))

    participant_scores.append({'pName': str(p_name), 'targetAnc': target_anc, 'targetID': str(target_id or ''), 'PM': int(pm), 'HC_T': int(hc_t), 'DR': float(dr), 'BR': int(br), 'NS': int(hh), 'ST_str': st_str, 'cssFinal': float(css), 'DOCS': float(docs_score), 'ANCHOR': float(anchor), 'diagLog': diag_log, 'AX': 1, 'CC': 1.0, 'BC': 1.0, 'DC': 1.0, 'TP': 1.0, 'isGroup': False, 'GD': len(spine_ids)})

anc_data = {}; part_data = {}
for lbl, grp in df.groupby('Dir_Label'):
    if str(lbl).strip() == 'No Matches' or not str(lbl).strip(): continue
    u_t = len(grp['Kit_Name'].unique())
    anc_data[str(lbl)] = {"name": str(lbl), "matches": len(grp), "cm": int(grp['cM'].sum()), "badge": "Platinum" if len(grp)>=30 else "Gold", "integrity": min(100, (len(grp)*2) + (u_t*10)), "testers": u_t}

for kname, grp in df.groupby('Kit_Name'):
    val_grp = grp[grp['Dir_Label'] != 'No Matches']
    p_score = next((i for i in participant_scores if i["pName"] == str(kname)), None)
    part_data[str(kname)] = {
        "name": str(kname), "sort_key": str(grp.iloc[0].get('Tester_Sort_Key', str(kname).lower())).strip(),
        "matches": len(val_grp), "cm": int(val_grp['cM'].sum()) if not val_grp.empty else 0,
        "badge": "Keystone Tester" if len(val_grp)>=15 else "Study Participant",
        "css_status": p_score['ST_str'] if p_score else "UNSCORED", "ns": int(p_score['NS']) if p_score else 0,
        "br": int(p_score['BR']) if p_score else 0, "target_id": p_score['targetID'] if p_score else '',
        "docs_score": float(p_score['DOCS']) if p_score else 0.0, "diag_log": p_score['diagLog'] if p_score else [],
        "kit_code": str(grp.iloc[0]['Kit_Code']), "integrity": 95
    }

# ✨ THE FIX: INJECTING THE ZERO-MATCH PARTICIPANTS INTO THE SYSTEM!
for code, td in csv_auth.items():
    t_disp = f"{td['name']} [{td['id']}]" if td['id'] else f"{td['name']} [{code}]"
    if t_disp not in part_data:
        part_data[t_disp] = {
            "name": t_disp, "sort_key": td["sort_key"] or td["name"].lower(),
            "matches": 0, "cm": 0, "badge": "Study Participant",
            "css_status": "UNSCORED", "ns": 0, "br": 0, "target_id": "",
            "docs_score": 0.0, "diag_log": ["No correlated matches found in GEDCOM."],
            "kit_code": code, "integrity": 0
        }

# --- STEP 5: SAVE GLOBALS TO DISK ---
print("    [+] Saving Compiled JS_GLOBALS to Disk...")
db_json = df[['Dir_Label', 'Kit_Name', 'cM', 'Match_ID', 'Lineage', 'search_ids', 'search_names', 't_names', 't_ids', 'Tester_ID', 'Kit_Code']].rename(columns={'Dir_Label':'ancestor', 'Kit_Name':'participant', 'cM':'cm', 'Match_ID':'id', 'Lineage':'lineage', 'Tester_ID':'tester_id', 'Kit_Code':'kit_code'}).to_dict(orient='records')
final_json = {"PRECOMPUTED": participant_scores, "DATA": {"ancestors": anc_data, "participants": part_data, "persons": persons_data}, "DB": db_json}

def safe_json_encoder(obj):
    if pd.isna(obj): return None
    if hasattr(obj, 'item'): return obj.item()
    return str(obj)

with open(JSON_DB, "w") as f:
    json.dump(final_json, f, default=safe_json_encoder)

print(f"\n[SUCCESS] Cell 2 Complete. Data successfully crunched and saved to {JSON_DB}.")

      [CELL 2] DATA & MATH ENGINE STARTING...
    [+] Active GEDCOM: yates_study_2025.ged
    [+] Tracing Lineages & Generating CSV...
    [+] Calculating Math Matrices & Audits...
    [+] Saving Compiled JS_GLOBALS to Disk...

[SUCCESS] Cell 2 Complete. Data successfully crunched and saved to compiled_database.json.


In [18]:
# @title [CELL 4] The Vault Fetcher (Hub Integration)
import urllib.request

print("="*60)
print("      📥 DOWNLOADING VAULT FRAGMENTS FROM HUB...")
print("="*60)

# The 22 fragments we just stored in London
fragment_keys = [
    'css_base', 'nav', 'site_info', 'js_core', 'btt_btn', 'proof_engine_tmpl',
    'dossier_tmpl', 'anchor_content', 'anchor_theory', 'anchor_css', 'network_page',
    'consolidator_html', 'consolidator_js', 'consolidator_css', 'admin_css',
    'register_css', 'glossary_inline', 'gedmatch_inline', 'contents_content',
    'share_content', 'subscribe_content', 'theory_page'
]

SITE_TEMPLATES = {}
hub_url = "https://yates.one-name.net/anchor/anchor-hub/"

print("[+] Reaching into London Hub for text blocks...")

# Disguise as a browser to bypass the 403 Firewall
req_headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36'}

for key in fragment_keys:
    try:
        req = urllib.request.Request(f"{hub_url}{key}.txt", headers=req_headers)
        SITE_TEMPLATES[key] = urllib.request.urlopen(req).read().decode('utf-8')
    except Exception as e:
        print(f"    [-] Missing Fragment: {key}.txt ({e})")
        SITE_TEMPLATES[key] = f""

print(f"    ✅ Successfully loaded {len(SITE_TEMPLATES)} fragments into Engine Memory!")
print("\n🎉 Vault Fetcher Complete. Ready for Publisher.")

      📥 DOWNLOADING VAULT FRAGMENTS FROM HUB...
[+] Reaching into London Hub for text blocks...
    ✅ Successfully loaded 22 fragments into Engine Memory!

🎉 Vault Fetcher Complete. Ready for Publisher.


In [25]:
# @title [CELL 5] The Jinja2 Publisher
def run_publisher():
    print("="*60)
    print("      [CELL 5] PUBLISHER STARTING (Jinja2 Engine Active)...")
    print("="*60)

    import os, json, pytz, urllib.request
    import pandas as pd
    from jinja2 import Template
    from datetime import datetime
    from ftplib import FTP_TLS
    try:
        from google.colab import userdata
    except ImportError:
        pass

    # 🚨 THE MEMORY SHIELD 🚨
    if 'SITE_TEMPLATES' not in globals() or 'anchor_theory' not in SITE_TEMPLATES:
        print("\n❌ STOPPING PUBLISHER: Colab Memory Wipe Detected! ❌")
        return print("    👉 FIX: Please scroll up and hit 'Play' on CELL 3 and CELL 4 to reload your text, then try this again.")

    try:
        HOST = os.environ.get("FTP_HOST") or userdata.get("FTP_HOST")
        USER = os.environ.get("FTP_USER") or userdata.get("FTP_USER")
        PASS = os.environ.get("FTP_PASS") or userdata.get("FTP_PASS")
    except Exception as e:
        return print(f"❌ Credential Error: {e}")

    JSON_DB = "compiled_database.json"
    CSV_DB = "engine_database.csv"

    if not os.path.exists(JSON_DB):
        return print("❌ ERROR: compiled_database.json not found. Run Cell 2 first.")

    print("    [+] Loading Pre-Calculated Data...")
    with open(JSON_DB, "r") as f:
        master_data = json.load(f)

    # 1. BUILD THE DATABASE JS FILE
    pre_str = json.dumps(master_data.get('PRECOMPUTED', []))
    data_str = json.dumps(master_data.get('DATA', {}))
    db_str = json.dumps(master_data.get('DB', []))

    js_file_content = f"window.PRECOMPUTED={pre_str};\nwindow.DATA={data_str};\nwindow.DB={db_str};\n"
    cache_buster = int(datetime.now().timestamp())
    db_script_tag = f'<script src="database.js?v={cache_buster}"></script>'

    df = pd.read_csv(CSV_DB, encoding="iso-8859-15")
    df.fillna('', inplace=True)
    df.replace('nan', '', inplace=True)

    if "Authority_Directory_Label" in df.columns:
        df.rename(columns={"Authority_Directory_Label": "Dir_Label", "Tester_Code": "Kit_Code", "Match_Lineage": "Lineage", "Match_Path_IDs": "s_ids", "Tester_Display": "Kit_Name"}, inplace=True)

    print("    [+] Building Data Tables...")
    df_valid = df[df['Dir_Label'] != 'No Matches'].copy()

    def get_sort_key(kit_name):
        return str(master_data['DATA']['participants'].get(kit_name, {}).get('sort_key', kit_name)).lower()

    df_valid['sort_key'] = df_valid['Kit_Name'].apply(get_sort_key)
    mc = df_valid['Dir_Label'].value_counts()

    def normalize_id(val): return f"I{str(val).replace('@', '').strip()}" if str(val).replace('@', '').strip().isdigit() else str(val).replace('@', '').strip()

    def format_reg(r):
        m_id, m_name, d_label, kit_name, cm_val = str(r.get("Match_ID", "")), str(r.get("Match_Name", "")), str(r.get("Dir_Label", "")).split('(')[0].strip(), str(r["Kit_Name"]), str(r["cM"])
        lin_len = len(str(r.get("Lineage", "")).split("->"))
        text = f"<b>{kit_name}</b> is a {cm_val} cM match to <a href='https://yates.one-name.net/tng/verticalchart.php?personID={normalize_id(m_id)}&tree=tree1&parentset=0&display=vertical&generations=15' target='_blank'><b>{m_name}</b></a> via {d_label} back {lin_len} generations."
        if mc.get(r['Dir_Label'], 0) == 1: return f"<div style='background-color: #fffde7; padding: 12px; margin: -12px; border-left: 5px solid #fbc02d;'>{text} <span style='float:right; font-size:0.85em; color:#e65100; font-weight:bold; background:#fff8e1; padding:3px 8px; border-radius:4px; border:1px solid #ffe082;'>🌟 Singleton Line</span></div>"
        return f"<div style='padding: 12px; margin: -12px;'>{text}</div>"

    def format_tree(r):
        m_id, m_name, lin_str, kit_name = str(r.get("Match_ID", "")), str(r.get("Match_Name", "")), str(r.get("Lineage", "")), str(r["Kit_Name"])
        linked_lin = lin_str.replace(m_name, f'<a href="https://yates.one-name.net/tng/verticalchart.php?personID={normalize_id(m_id)}&tree=tree1&parentset=0&display=vertical&generations=15" target="_blank" style="color:#006064;text-decoration:none;font-weight:bold;">{m_name}</a>') if m_name in lin_str else lin_str
        text = f"<b style='color:#4a148c;'>{kit_name}</b>: {linked_lin}"
        if mc.get(r['Dir_Label'], 0) == 1: return f"<div style='background-color: #fffde7; padding: 12px; margin: -12px; border-left: 5px solid #fbc02d;'>{text} <span style='float:right; font-size:0.85em; color:#e65100; font-weight:bold; background:#fff8e1; padding:3px 8px; border-radius:4px; border:1px solid #ffe082;'>🌟 Singleton Line</span></div>"
        return f"<div style='padding: 12px; margin: -12px;'>{text}</div>"

    df_valid['Reg_Narrative'] = df_valid.apply(format_reg, axis=1)
    df_valid['Tree_Narrative'] = df_valid.apply(format_tree, axis=1)

    df_reg_za = df_valid.sort_values(by=['Dir_Label', 'sort_key'], ascending=[False, True]).copy()
    df_reg_za.rename(columns={'Reg_Narrative': 'Participants who tested-Who they matched-Oldest known Yates ancestor'}, inplace=True)
    df_reg_az = df_valid.sort_values(by=['sort_key', 'Dir_Label'], ascending=[True, False]).copy()
    df_reg_az.rename(columns={'Reg_Narrative': 'Participants who tested-Who they matched-Oldest known Yates ancestor'}, inplace=True)

    df_tree_za = df_valid.sort_values(by=['Dir_Label', 'sort_key'], ascending=[False, True]).copy()
    df_tree_za.rename(columns={'Tree_Narrative': 'Visual Lineage Path'}, inplace=True)
    df_tree_az = df_valid.sort_values(by=['sort_key', 'Dir_Label'], ascending=[True, False]).copy()
    df_tree_az.rename(columns={'Tree_Narrative': 'Visual Lineage Path'}, inplace=True)

    toggle_reg_za = f'<div class="no-print" style="text-align:center; margin:15px auto; max-width:1400px; padding:10px; background:#e0f7fa; border:1px solid #b2ebf2; border-radius:4px; font-family:sans-serif; font-size:14px;"><strong>Sort Register:</strong> &nbsp;<span style="color:#006064; font-weight:bold;">By Ancestral Line (Z-A)</span> &nbsp;|&nbsp; <a href="ons_yates_dna_register_participants.shtml" style="color:#00acc1; text-decoration:none;">By Participant (A-Z)</a></div>'
    toggle_reg_az = f'<div class="no-print" style="text-align:center; margin:15px auto; max-width:1400px; padding:10px; background:#e0f7fa; border:1px solid #b2ebf2; border-radius:4px; font-family:sans-serif; font-size:14px;"><strong>Sort Register:</strong> &nbsp;<a href="ons_yates_dna_register.shtml" style="color:#00acc1; text-decoration:none;">By Ancestral Line (Z-A)</a> &nbsp;|&nbsp; <span style="color:#006064; font-weight:bold;">By Participant (A-Z)</span></div>'
    toggle_tree_za = f'<div class="no-print" style="text-align:center; margin:15px auto; max-width:1400px; padding:10px; background:#e0f7fa; border:1px solid #b2ebf2; border-radius:4px; font-family:sans-serif; font-size:14px;"><strong>Sort Trees:</strong> &nbsp;<span style="color:#006064; font-weight:bold;">By Ancestral Line (Z-A)</span> &nbsp;|&nbsp; <a href="just-trees-az.shtml" style="color:#00acc1; text-decoration:none;">By Participant (A-Z)</a></div>'
    toggle_tree_az = f'<div class="no-print" style="text-align:center; margin:15px auto; max-width:1400px; padding:10px; background:#e0f7fa; border:1px solid #b2ebf2; border-radius:4px; font-family:sans-serif; font-size:14px;"><strong>Sort Trees:</strong> &nbsp;<a href="just-trees.shtml" style="color:#00acc1; text-decoration:none;">By Ancestral Line (Z-A)</a> &nbsp;|&nbsp; <span style="color:#006064; font-weight:bold;">By Participant (A-Z)</span></div>'

    est = pytz.timezone('US/Eastern')
    timestamp = datetime.now(est).strftime("%B %d, %Y %I:%M %p %Z")
    num_participants = len(master_data['DATA']['participants'])
    num_matches = len(df_valid)
    stats_bar_full = f'<div style="background:#f4f4f4;border-top:1px solid #ddd;border-bottom:1px solid #ddd;font-family:sans-serif;font-size:14px;color:#555;padding:10px 15px;text-align:center;margin-bottom:0;"><strong>Study Data Current As Of:</strong> {timestamp} | <strong>Total Participants:</strong> {num_participants} | <strong>Total Autosomal matches:</strong> {num_matches:,}</div>'

    current_year = str(datetime.now(est).year)
    LEGAL_FOOTER = r'''<div class="legal-footer no-print" style="margin-top:50px;padding:20px;background:#f4f4f4;border-top:1px solid #ddd;text-align:center;color:#666;font-family:sans-serif;font-size:0.85em;clear:both;"><p style="margin-bottom:5px;font-size:1.1em;color:#333;"><strong>&copy; __YEAR__ Ronald Eugene Yates. All Rights Reserved.</strong></p><p style="margin-bottom:5px;">Generated by <em>The Forensic Genealogy Publisher&trade;</em></p><p style="font-style:italic;color:#888;margin-bottom:0;max-width:800px;margin-left:auto;margin-right:auto;">The terms "Forensic Handshake", "Brick Wall Buster", and "Collateral Saturation" are trademarks of Ronald Eugene Yates.</p></div>'''.replace('__YEAR__', current_year)

    # ---------------------------------------------------------
    # 2. JINJA2 TEMPLATE ENGINE (Replaces the old make_page)
    # ---------------------------------------------------------
    print("    [+] Fetching Jinja Templates from London Hub...")
    pages_to_upload = {}

    req_headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36'}

    try:
        # Fetch Master Template
        req_master = urllib.request.Request("https://yates.one-name.net/anchor/anchor-hub/template_master.html", headers=req_headers)
        master_jinja = Template(urllib.request.urlopen(req_master).read().decode('utf-8'))

        # Fetch Admin Hub Template
        req_admin = urllib.request.Request("https://yates.one-name.net/anchor/anchor-hub/template_admin.html", headers=req_headers)
        admin_jinja = Template(urllib.request.urlopen(req_admin).read().decode('utf-8'))

        # Fetch Zero Matches Template
        req_zm = urllib.request.Request("https://yates.one-name.net/anchor/anchor-hub/template_zero_matches.html", headers=req_headers)
        zm_jinja = Template(urllib.request.urlopen(req_zm).read().decode('utf-8'))
    except Exception as e:
        return print(f"❌ JINJA ERROR: Failed to download templates from London. Error: {e}")

    def render_master(title, content, nav_b, bar, extra_css=""):
        s_info = SITE_TEMPLATES.get('site_info', '') if nav_b else ""
        content_with_db = content.replace('__DATABASE_SCRIPT__', db_script_tag)
        return master_jinja.render(
            title=title, stats_bar=bar, nav_html=SITE_TEMPLATES.get('nav', ''),
            site_info=s_info, main_content=content_with_db, extra_css=extra_css,
            legal_footer=LEGAL_FOOTER, js_core=SITE_TEMPLATES.get('js_core', ''), btt_btn=SITE_TEMPLATES.get('btt_btn', '')
        )

    print("    [+] Mass-Assembling Pages via Jinja2...")

    # A. Special Pages (Dossier & Proof Engine)
    pages_to_upload["proof_engine.html"] = SITE_TEMPLATES.get('proof_engine_tmpl', '').replace('__CSS_BASE__', SITE_TEMPLATES.get('css_base', '')).replace('__STATS_BAR__', stats_bar_full).replace('__NAV_HTML__', SITE_TEMPLATES.get('nav', '')).replace('__DATABASE_SCRIPT__', db_script_tag).replace('__LEGAL_FOOTER__', LEGAL_FOOTER)
    pages_to_upload["dna_dossier.html"] = SITE_TEMPLATES.get('dossier_tmpl', '').replace('__CSS_BASE__', SITE_TEMPLATES.get('css_base', '')).replace('__STATS_BAR__', stats_bar_full).replace('__NAV_HTML__', SITE_TEMPLATES.get('nav', '')).replace('__DATABASE_SCRIPT__', db_script_tag).replace('__LEGAL_FOOTER__', LEGAL_FOOTER)

    # B. Pure Jinja Rendered Pages
    pages_to_upload["research_admin.html"] = admin_jinja.render(title="Yates Research Admin Hub", stats_bar=stats_bar_full, nav_html=SITE_TEMPLATES.get('nav', ''), database_script=db_script_tag, legal_footer=LEGAL_FOOTER, js_core=SITE_TEMPLATES.get('js_core', ''), btt_btn=SITE_TEMPLATES.get('btt_btn', ''), extra_css=SITE_TEMPLATES.get('admin_css', ''))
    pages_to_upload["zero_matches.html"] = zm_jinja.render(title="Zero Matches Audit", stats_bar=stats_bar_full, nav_html=SITE_TEMPLATES.get('nav', ''), database_script=db_script_tag, legal_footer=LEGAL_FOOTER, js_core=SITE_TEMPLATES.get('js_core', ''), btt_btn=SITE_TEMPLATES.get('btt_btn', ''))

    # C. Master Template Routed Pages
    anchor_injected = SITE_TEMPLATES.get('anchor_content', '').replace('__TAB_FRAMEWORK__', SITE_TEMPLATES.get('anchor_theory', ''))
    pages_to_upload["anchor.html"] = render_master("ANCHOR Database", anchor_injected, False, stats_bar_full, SITE_TEMPLATES.get('anchor_css', ''))
    pages_to_upload["dna_network.html"] = render_master("DNA Network Visualizer", SITE_TEMPLATES.get('network_page', ''), False, stats_bar_full, "")
    consol_html = SITE_TEMPLATES.get('consolidator_html', '') + SITE_TEMPLATES.get('consolidator_js', '')
    pages_to_upload["proof_consolidator.html"] = render_master("Master Proof Report", consol_html, False, stats_bar_full, SITE_TEMPLATES.get('consolidator_css', ''))
    pages_to_upload["ons_yates_dna_register.shtml"] = render_master("ONS Yates Study DNA Register", toggle_reg_za + f'<div class="table-scroll-wrapper">{df_reg_za.to_html(columns=["Participants who tested-Who they matched-Oldest known Yates ancestor"], index=False, border=1, classes="dataframe sortable", escape=False, table_id="reg-table")}</div>', True, stats_bar_full, SITE_TEMPLATES.get('register_css', ''))
    pages_to_upload["ons_yates_dna_register_participants.shtml"] = render_master("ONS Yates Study DNA Register", toggle_reg_az + f'<div class="table-scroll-wrapper">{df_reg_az.to_html(columns=["Participants who tested-Who they matched-Oldest known Yates ancestor"], index=False, border=1, classes="dataframe sortable", escape=False, table_id="reg-table")}</div>', True, stats_bar_full, SITE_TEMPLATES.get('register_css', ''))
    pages_to_upload["just-trees.shtml"] = render_master("Ancestor Register (Trees View)", toggle_tree_za + f'<div class="table-scroll-wrapper">{df_tree_za[["Visual Lineage Path"]].to_html(index=False, border=1, classes="dataframe sortable", escape=False, table_id="reg-table")}</div>', True, stats_bar_full, SITE_TEMPLATES.get('register_css', ''))
    pages_to_upload["just-trees-az.shtml"] = render_master("Ancestor Register (Trees View)", toggle_tree_az + f'<div class="table-scroll-wrapper">{df_tree_az[["Visual Lineage Path"]].to_html(index=False, border=1, classes="dataframe sortable", escape=False, table_id="reg-table")}</div>', True, stats_bar_full, SITE_TEMPLATES.get('register_css', ''))
    pages_to_upload["data_glossary.shtml"] = render_master("Data Glossary", SITE_TEMPLATES.get('glossary_inline', ''), False, stats_bar_full, "")
    pages_to_upload["gedmatch_integration.shtml"] = render_master("GEDmatch Hub", SITE_TEMPLATES.get('gedmatch_inline', ''), False, stats_bar_full, "")
    pages_to_upload["contents.shtml"] = render_master("Yates Study User Guide", SITE_TEMPLATES.get('contents_content', ''), False, stats_bar_full, "")
    pages_to_upload["share_dna.shtml"] = render_master("Share Your Ancestry DNA Matches", SITE_TEMPLATES.get('share_content', ''), False, stats_bar_full, "")
    pages_to_upload["subscribe.shtml"] = render_master("Join the Yates Research Community", SITE_TEMPLATES.get('subscribe_content', ''), False, stats_bar_full, "")
    pages_to_upload["dna_theory_of_the_case.htm"] = render_master("Theory of the Case", SITE_TEMPLATES.get('theory_page', ''), False, stats_bar_full, "")

    # D. URL Aliases
    pages_to_upload["anchor_frame.htm"] = pages_to_upload["anchor.html"]
    pages_to_upload["dna_network.shtml"] = pages_to_upload["dna_network.html"]
    pages_to_upload["admin_singletons.shtml"] = pages_to_upload["ons_yates_dna_register.shtml"]
    pages_to_upload["admin_singletons_participants.shtml"] = pages_to_upload["ons_yates_dna_register_participants.shtml"]
    pages_to_upload["yates_ancestor_register.shtml"] = pages_to_upload["ons_yates_dna_register.shtml"]

    # --- AUTO-CORRECTOR: Update old links to new Anchor directory ---
    print("    [*] Scanning and updating old header links to the new Anchor directory...")
    for fn in pages_to_upload:
        pages_to_upload[fn] = pages_to_upload[fn].replace("https://yates.one-name.net/ons-study/", "https://yates.one-name.net/anchor/anc-1000-yates/")
        pages_to_upload[fn] = pages_to_upload[fn].replace("/ons-study/", "/anchor/anc-1000-yates/")
        pages_to_upload[fn] = pages_to_upload[fn].replace('href="ons-study/', 'href="anchor/anc-1000-yates/')

    # 3. WRITE AND UPLOAD FILES
    if not os.path.exists('site'): os.makedirs('site')

    print("\n[LOCAL] Overwriting Files...")
    with open("site/database.js", "w", encoding="utf-8") as f: f.write(js_file_content)

    for fn, content in pages_to_upload.items():
        local_path = os.path.join('site', fn)
        if os.path.exists(local_path): os.remove(local_path)
        with open(local_path, "w", encoding="utf-8") as f: f.write(content)

    print("\n[STEP 3] Uploading via FTP to Live Server...")
    upload_success = 0
    upload_fails = 0
    failed_files = []

    try:
        ftps = FTP_TLS()
        ftps.connect(HOST, 21); ftps.auth(); ftps.login(USER, PASS); ftps.prot_p()
        ftps.cwd("anchor/anc-1000-yates")

        files_to_upload = list(pages_to_upload.keys()) + ["database.js"]
        if os.path.exists(CSV_DB):
            import shutil
            shutil.copyfile(CSV_DB, os.path.join('site', CSV_DB))
            files_to_upload.append(CSV_DB)

        # The Bulletproof Loop
        for i, fn in enumerate(files_to_upload, 1):
            local_path = os.path.join('site', fn)
            try:
                with open(local_path, "rb") as fh:
                    ftps.storbinary(f"STOR {fn}", fh)
                upload_success += 1
                print(f"    [{i}/{len(files_to_upload)}] 📤 Success: {fn}")
            except Exception as file_e:
                upload_fails += 1
                failed_files.append(fn)
                print(f"    [{i}/{len(files_to_upload)}] ❌ FAILED: {fn} ({file_e})")

        ftps.quit()

        # The Highly Specific Final Status Report
        print("\n" + "="*60)
        if upload_fails == 0:
            print(f"✅ PUBLISHER COMPLETE: All {upload_success} files successfully published to London.")
            print("    🌐 View the new site here: https://yates.one-name.net/anchor/anc-1000-yates/")
        else:
            print(f"⚠️ PUBLISHER FINISHED WITH ERRORS: {upload_success} uploaded, {upload_fails} failed.")
            print(f"   Failed files to check: {', '.join(failed_files)}")
        print("="*60)

    except Exception as e:
        print(f"\n❌ CRITICAL FTP ERROR: Could not connect to the server at all. {e}")

# EXECUTE THE PUBLISHER
run_publisher()

      [CELL 5] PUBLISHER STARTING (Jinja2 Engine Active)...
    [+] Loading Pre-Calculated Data...


/tmp/ipykernel_270/629552083.py:49: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.fillna('', inplace=True)


    [+] Building Data Tables...
    [+] Fetching Jinja Templates from London Hub...
    [+] Mass-Assembling Pages via Jinja2...
    [*] Scanning and updating old header links to the new Anchor directory...

[LOCAL] Overwriting Files...

[STEP 3] Uploading via FTP to Live Server...
    [1/24] 📤 Success: proof_engine.html
    [2/24] 📤 Success: dna_dossier.html
    [3/24] 📤 Success: research_admin.html
    [4/24] 📤 Success: zero_matches.html
    [5/24] 📤 Success: anchor.html
    [6/24] 📤 Success: dna_network.html
    [7/24] 📤 Success: proof_consolidator.html
    [8/24] 📤 Success: ons_yates_dna_register.shtml
    [9/24] 📤 Success: ons_yates_dna_register_participants.shtml
    [10/24] 📤 Success: just-trees.shtml
    [11/24] 📤 Success: just-trees-az.shtml
    [12/24] 📤 Success: data_glossary.shtml
    [13/24] 📤 Success: gedmatch_integration.shtml
    [14/24] 📤 Success: contents.shtml
    [15/24] 📤 Success: share_dna.shtml
    [16/24] 📤 Success: subscribe.shtml
    [17/24] 📤 Success: dna_theo

In [20]:
# @title [CELL 6] The Time Machine (Smart Archiver & Backup)
def run_archiver():
    print("="*60)
    print("      [CELL] THE TIME MACHINE (Archive & Sync)")
    print("="*60)

    import zipfile
    import os
    import pytz
    import json
    from datetime import datetime
    from google.colab import files
    try:
        from google.colab import userdata
    except ImportError:
        pass

    est = pytz.timezone('US/Eastern')
    timestamp = datetime.now(est).strftime("%Y-%m-%d_%H%M_%S")

    # 🌟 Detect exactly where the HTML files are stored (Cell 5 puts them in 'site')
    target_dir = 'site' if os.path.exists('site') else '.'

    # --- 1. GENERATE SITE SNAPSHOT JSON ---
    print("[STEP 1] Generating Site Snapshot JSON...")
    snapshot_data = {}

    # Grab files from the correct directory
    html_files = [f for f in os.listdir(target_dir) if f.lower().endswith(('.shtml', '.html'))]

    if not html_files:
        print(f"    ❌ No generated HTML files found in '{target_dir}'! Run Cell 5 first.")
        return

    for f_name in html_files:
        try:
            with open(os.path.join(target_dir, f_name), 'r', encoding='utf-8') as fh:
                snapshot_data[f_name] = fh.read()
        except Exception as e:
            print(f"    ⚠️ Could not read {f_name}: {e}")

    snapshot_name = f"site_snapshot_{timestamp}.json"
    with open(snapshot_name, 'w', encoding='utf-8') as f:
        json.dump(snapshot_data, f)
    print(f"    ✅ Created snapshot JSON: {snapshot_name}")

    # --- 2. CREATE MASTER ZIP VAULT ---
    extensions = ('.csv', '.shtml', '.html', '.json', '.js', '.css', '.ged')
    files_to_pack = []

    # Smart walk to grab files from ALL relevant Colab folders
    for root_dir, dirs, files_in_dir in os.walk('.'):
        if '.config' in root_dir or 'sample_data' in root_dir: continue # Skip Colab system files
        for f in files_in_dir:
            if f.lower().endswith(extensions):
                files_to_pack.append(os.path.join(root_dir, f))

    zip_name = f"Yates_Study_Backup_{timestamp}.zip"

    print(f"\n[STEP 2] Compressing {len(files_to_pack)} files into {zip_name}...")
    try:
        with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
            for file in files_to_pack:
                # Keep the folder structure clean inside the zip
                arcname = file.replace('./', '') if file.startswith('./') else file
                zf.write(file, arcname)
        print(f"    ✅ Archive Created: {zip_name} ({os.path.getsize(zip_name)/1024/1024:.2f} MB)")
    except Exception as e:
        print(f"    ❌ Compression Failed: {e}")
        return

    # --- 3. FTP UPLOAD (NEW ANCHOR BACKUPS FOLDER) ---
    print("\n[STEP 3] Uploading to Web Server Vault (FTP)...")
    try:
        from ftplib import FTP_TLS
        HOST = os.environ.get("FTP_HOST") or userdata.get("FTP_HOST")
        USER = os.environ.get("FTP_USER") or userdata.get("FTP_USER")
        PASS = os.environ.get("FTP_PASS") or userdata.get("FTP_PASS")

        ftps = FTP_TLS()
        ftps.connect(HOST, 21); ftps.auth(); ftps.login(USER, PASS); ftps.prot_p()

        # Route directly to the new Spoke architecture
        ftps.cwd("anchor/anc-1000-yates")

        # Safely check for or create the backups folder
        if "backups" not in ftps.nlst():
            ftps.mkd("backups")
        ftps.cwd("backups")

        with open(zip_name, "rb") as fh:
            ftps.storbinary(f"STOR {zip_name}", fh)
        print(f"    ✅ FTP Success: /anchor/anc-1000-yates/backups/{zip_name}")

        with open(snapshot_name, "rb") as fh:
            ftps.storbinary(f"STOR {snapshot_name}", fh)
        print(f"    ✅ FTP Success: /anchor/anc-1000-yates/backups/{snapshot_name}")

        ftps.quit()
    except Exception as e:
        print(f"    ⚠️ FTP Upload skipped/failed: {e}")

    # --- 4. DROPBOX SYNC (REFRESH TOKEN METHOD) ---
    print("\n[STEP 4] Syncing to Dropbox...")
    try:
        dbx_app_key = os.environ.get("DBX_APP_KEY") or userdata.get("DBX_APP_KEY")
        dbx_app_secret = os.environ.get("DBX_APP_SECRET") or userdata.get("DBX_APP_SECRET")
        dbx_refresh = os.environ.get("DBX_REFRESH_TOKEN") or userdata.get("DBX_REFRESH_TOKEN")

        if not dbx_refresh:
            print("    ❌ ERROR: 'DBX_REFRESH_TOKEN' not found in Colab Secrets.")
        else:
            try:
                import dropbox
            except ImportError:
                os.system('pip install dropbox')
                import dropbox

            dbx = dropbox.Dropbox(
                app_key=dbx_app_key,
                app_secret=dbx_app_secret,
                oauth2_refresh_token=dbx_refresh
            )

            target_path = f"/Yates_Study_Sync/archives/{snapshot_name}"

            with open(snapshot_name, "rb") as f:
                dbx.files_upload(f.read(), target_path)
            print(f"    ✅ Dropbox Sync Success: {target_path}")

    except Exception as e:
        print(f"    ❌ Dropbox Upload Failed: {e}")

    # --- 5. TRIGGER LOCAL DOWNLOAD ---
    print("\n[STEP 5] Triggering Local Download...")
    try:
        files.download(zip_name)
        print("    ✅ Please check your browser downloads for the Archive.")
    except:
        print("    ⚠️ Could not auto-download. You can download the zip manually from the Colab files pane.")

# Execute the function immediately
run_archiver()

      [CELL] THE TIME MACHINE (Archive & Sync)
[STEP 1] Generating Site Snapshot JSON...
    ✅ Created snapshot JSON: site_snapshot_2026-03-02_1948_45.json

[STEP 2] Compressing 29 files into Yates_Study_Backup_2026-03-02_1948_45.zip...
    ✅ Archive Created: Yates_Study_Backup_2026-03-02_1948_45.zip (25.17 MB)

[STEP 3] Uploading to Web Server Vault (FTP)...
    ✅ FTP Success: /anchor/anc-1000-yates/backups/Yates_Study_Backup_2026-03-02_1948_45.zip
    ✅ FTP Success: /anchor/anc-1000-yates/backups/site_snapshot_2026-03-02_1948_45.json

[STEP 4] Syncing to Dropbox...
    ✅ Dropbox Sync Success: /Yates_Study_Sync/archives/site_snapshot_2026-03-02_1948_45.json

[STEP 5] Triggering Local Download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

    ✅ Please check your browser downloads for the Archive.
